# LSC5 | Reconocimiento de señas aisladas con landmarks y TCN

**Objetivo:** desarrollar una solución reproducible para clasificar las cinco señas de LSC5, priorizando la generalización a una persona no observada durante el entrenamiento.

## Estrategia elegida

1. Auditar los archivos y la distribución de clases, participantes y vistas.
2. Transformar los landmarks de MediaPipe Holistic en características geométricas y temporales compactas.
3. Medir una línea base SVM.
4. Entrenar un **TCN multistream compacto** con codificadores independientes para manos, cuerpo y rostro, fusionados antes del bloque temporal.
5. Validar con **Leave-One-Signer-Out (LOSO)**: un participante completo queda fuera en cada fold. Dentro de los dos participantes de entrenamiento se usa una partición agrupada por ejecución para escoger el número de épocas. Después se reentrena con los dos participantes completos y se evalúa una sola vez en el participante reservado.
6. Entrenar un modelo final con los tres participantes públicos usando la mediana de las mejores épocas de LOSO.

> **Decisión de alcance:** en esta primera versión no procesamos RGB. Los landmarks ya están disponibles y permiten concentrar el tiempo en validación rigurosa y modelado temporal. La fusión con video solo se justificaría después de demostrar que un modelo visual aporta una mejora real en LOSO.

## Reglas de integridad

- El test no se carga ni se inspecciona durante exploración, desarrollo, selección de características o validación.
- No se calculan estadísticas de normalización con test.
- El bloque opcional de inferencia final está separado y desactivado por defecto. Solo se ejecuta cuando el modelo y sus hiperparámetros estén congelados.
- No se usan etiquetas, metadatos privados ni retroalimentación del test para modificar el modelo.

**Métrica principal:** Accuracy Top-1. También registramos macro-F1 y matrices de confusión como diagnóstico.

Fuentes de referencia: Manual LSC5 para estudiantes (octubre de 2026), Compromiso ético y uso responsable del dataset y página oficial del reto: https://www.latamiachallenge.com/dataset.php.

## 0. Configuración del experimento

En Kaggle, agrega el dataset oficial mediante **Add Input**. Normalmente quedará montado bajo `/kaggle/input/`.

Si la detección automática no encuentra las rutas, define `TRAIN_LANDMARKS_DIR` manualmente. Debe apuntar a la carpeta `train/landmarks` que contiene `sign_01/` ... `sign_05/`.

**Configuración para Kaggle:** empieza con GPU T4/P100 si está disponible. En CPU se puede ejecutar el baseline y una prueba reducida del TCN.

In [ ]:
from pathlib import Path
import os, re, json, math, random, time, copy, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
from torch import nn
from IPython.display import display
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------
# Configuración editable
# -----------------------------
DATA_ROOT = Path('/kaggle/input')
TRAIN_LANDMARKS_DIR = None  # Ejemplo: Path('/kaggle/input/lsc5/train/landmarks')
TEST_LANDMARKS_DIR = None   # Se usa únicamente en la celda opcional de inferencia final.

SEED = 42
SEQ_LEN = 64
BATCH_SIZE = 32
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
TCN_CHANNELS = 96
DROPOUT = 0.25
NUM_WORKERS = 2
USE_AMP = True

# Debe permanecer False durante desarrollo/validación.
RUN_FINAL_TEST_INFERENCE = False

OUT_DIR = Path('/kaggle/working/lsc5_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [f'sign_{i:02d}' for i in range(1, 6)]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def seed_everything(seed=SEED):
    """Fija semillas para hacer los experimentos lo más reproducibles posible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Se prioriza reproducibilidad; puede ser algo más lento que el modo benchmark.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print('PyTorch:', torch.__version__)
print('Dispositivo:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No se detectó GPU. Para el TCN, activa GPU en Kaggle si está disponible.')
print('Directorio de artefactos:', OUT_DIR)

## 1. Detección de rutas y construcción del inventario

El manual describe los archivos etiquetados en `train/landmarks/sign_XX/` y los nombres originales con el patrón `S##_M##_V##_C##_Clip####`. Extraemos el participante (`M##`) del nombre y usamos la carpeta `sign_XX` como etiqueta cuando esté disponible.

Para evitar que vistas simultáneas de una ejecución se separen en validación interna, creamos `group_id` usando la seña, el participante y `Clip####`, ignorando vista y cámara.

In [ ]:
def _class_dirs(path: Path):
    if not path.exists() or not path.is_dir():
        return []
    return [p for p in path.iterdir() if p.is_dir() and re.fullmatch(r'sign_\d{2}', p.name)]


def discover_train_landmarks(root=DATA_ROOT, override=None):
    """Encuentra train/landmarks sin entrar en carpetas de test."""
    if override is not None:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(f'La ruta TRAIN_LANDMARKS_DIR no existe: {p}')
        return p

    if not root.exists():
        raise FileNotFoundError(f'No existe DATA_ROOT: {root}')

    candidates = []
    for p in root.rglob('landmarks'):
        if not p.is_dir():
            continue
        parts_lower = [part.lower() for part in p.parts]
        if 'test' in parts_lower:
            continue
        classes = _class_dirs(p)
        if len(classes) >= 5 and any(p.rglob('*.npy')):
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError(
            'No encontré una carpeta train/landmarks con subcarpetas sign_01 ... sign_05. '
            'Define TRAIN_LANDMARKS_DIR manualmente.'
        )

    # Preferir una ruta que contenga explícitamente "train".
    candidates.sort(key=lambda p: ('train' not in [x.lower() for x in p.parts], len(str(p))))
    return candidates[0]


def parse_sample_metadata(path: Path):
    """Extrae etiqueta, participante, vista, cámara y grupo de ejecución."""
    stem = path.stem
    parent_match = re.fullmatch(r'(sign_\d{2})', path.parent.name)
    sign_match = re.search(r'(?:^|_)S(\d{2})(?:_|$)', stem)
    label = parent_match.group(1) if parent_match else (f'sign_{int(sign_match.group(1)):02d}' if sign_match else None)

    signer_match = re.search(r'(?:^|_)M(\d{2})(?:_|$)', stem)
    if signer_match is None:
        raise ValueError(f'No pude extraer participante M## del nombre: {path.name}')
    signer = f'M{signer_match.group(1)}'

    view_match = re.search(r'(?:^|_)V(\d{2})(?:_|$)', stem)
    camera_match = re.search(r'(?:^|_)C(\d{2})(?:_|$)', stem)
    clip_match = re.search(r'Clip(\d+)', stem, flags=re.IGNORECASE)

    if label not in CLASS_NAMES:
        raise ValueError(f'No pude identificar una clase sign_01 ... sign_05: {path}')

    # Agrupa las vistas/cámaras que comparten el mismo clip. Si no hay Clip, usa el archivo.
    group_id = f'{label}_{signer}_Clip{clip_match.group(1)}' if clip_match else stem
    return {
        'path': str(path),
        'video_id': stem,
        'label': label,
        'y': CLASS_TO_ID[label],
        'signer': signer,
        'view': f'V{view_match.group(1)}' if view_match else 'unknown',
        'camera': f'C{camera_match.group(1)}' if camera_match else 'unknown',
        'clip': clip_match.group(1) if clip_match else None,
        'group_id': group_id,
    }


TRAIN_LANDMARKS_DIR = discover_train_landmarks(DATA_ROOT, TRAIN_LANDMARKS_DIR)
records = []
for npy_path in sorted(TRAIN_LANDMARKS_DIR.rglob('*.npy')):
    # Esta carpeta debe ser exclusivamente el conjunto etiquetado de entrenamiento.
    records.append(parse_sample_metadata(npy_path))

metadata = pd.DataFrame(records)
if metadata.empty:
    raise RuntimeError('No se encontraron archivos .npy en TRAIN_LANDMARKS_DIR.')

print('Carpeta de landmarks:', TRAIN_LANDMARKS_DIR)
print('Muestras etiquetadas:', len(metadata))
print('Participantes:', sorted(metadata.signer.unique()))
print('Clases:', sorted(metadata.label.unique()))
display(metadata.head())

## 2. Auditoría del dataset

Antes de entrenar verificamos balance de clases, distribución por participante/vista, duplicados y correspondencia aproximada con los videos. No abrimos ni inspeccionamos el conjunto de test.

In [ ]:
# Resumen de distribución
class_counts = metadata['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
signer_class = pd.crosstab(metadata['signer'], metadata['label']).reindex(columns=CLASS_NAMES, fill_value=0)
view_counts = metadata['view'].value_counts(dropna=False)

print('Muestras por clase:')
display(class_counts.rename('n').to_frame())
print('Muestras por participante y clase:')
display(signer_class)
print('Muestras por vista:')
display(view_counts.rename('n').to_frame())

# Conteos esperados según el Manual LSC5 para el split público de entrenamiento.
if len(metadata) != 1125:
    warnings.warn(f'Se esperaban 1125 landmarks públicos y se encontraron {len(metadata)}. Verifica la ruta.')
if metadata['signer'].nunique() > 3:
    raise ValueError('Se detectaron más de tres participantes en train. No incluyas el participante reservado para test.')
if not (signer_class == 75).all().all():
    warnings.warn('La distribución no coincide exactamente con 75 muestras por clase y participante; revisa el inventario.')
print('Grupos de ejecución únicos:', metadata['group_id'].nunique())
print('Archivos repetidos por video_id:', int(metadata['video_id'].duplicated().sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
class_counts.plot(kind='bar', ax=axes[0], title='Muestras por clase')
axes[0].set_ylabel('Cantidad'); axes[0].tick_params(axis='x', rotation=30)
signer_class.plot(kind='bar', ax=axes[1], title='Clases por participante')
axes[1].set_ylabel('Cantidad'); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

# Comprobación de videos emparejados, si existe la carpeta hermana train/videos.
TRAIN_VIDEO_DIR = TRAIN_LANDMARKS_DIR.parent / 'videos'
if TRAIN_VIDEO_DIR.exists():
    video_stems = {p.stem for p in TRAIN_VIDEO_DIR.rglob('*.mp4')}
    metadata['has_video_pair'] = metadata['video_id'].isin(video_stems)
    print(f"Landmarks con video correspondiente: {metadata['has_video_pair'].sum()} / {len(metadata)}")
else:
    print('No encontré train/videos junto a train/landmarks; se continúa usando los .npy.')

# Inspección de forma y tipo en una muestra de archivos de cada clase.
shape_rows = []
for label in CLASS_NAMES:
    subset = metadata[metadata.label == label].head(3)
    for row in subset.itertuples():
        arr = np.load(row.path, allow_pickle=False)
        shape_rows.append({'label': label, 'file': Path(row.path).name,
                           'shape': arr.shape, 'dtype': str(arr.dtype),
                           'all_finite': bool(np.isfinite(arr).all())})
display(pd.DataFrame(shape_rows))

if metadata['signer'].nunique() != 3:
    warnings.warn(
        f"Se detectaron {metadata['signer'].nunique()} participantes etiquetados; "
        'el manual describe tres participantes públicos. Revisa que hayas agregado el split correcto.'
    )
if set(metadata['label'].unique()) != set(CLASS_NAMES):
    raise ValueError('Faltan clases. Revisa la ruta y la estructura del dataset.')

## 3. Ingeniería de características geométricas y temporales

Cada `.npy` debe contener una matriz `(n_fotogramas, 1662)`, según el manual: pose (132), rostro (1404), mano izquierda (63) y mano derecha (63).

Para reducir ruido y dimensionalidad sin descartar la información útil:

- **Manos:** 21 articulaciones por mano en coordenadas relativas a la muñeca.
- **Trayectoria:** posición de cada muñeca relativa al centro de hombros.
- **Cuerpo:** hombros, codos, muñecas y caderas superiores seleccionadas, normalizadas por el ancho de hombros.
- **Rostro:** un subconjunto pequeño de puntos de cejas, ojos, nariz y boca, relativo a la nariz y normalizado por distancia entre ojos.
- **Dinámica:** diferencias temporales (velocidad aproximada) de todas las características continuas.

La secuencia se remuestrea a 64 frames. Las estadísticas de estandarización se ajustan después, por separado dentro de cada partición de entrenamiento; nunca usando validación o test.

In [ ]:
# Índices de la representación descrita en el manual LSC5 / MediaPipe Holistic.
POSE_SLICE = slice(0, 132)
FACE_SLICE = slice(132, 1536)
LEFT_HAND_SLICE = slice(1536, 1599)
RIGHT_HAND_SLICE = slice(1599, 1662)

BODY_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24]  # hombros, codos, muñecas y caderas
FACE_JOINTS = [1, 33, 133, 362, 263, 61, 291, 13, 14, 70, 300]
BASE_FEATURE_DIM = 200
FEATURE_DIM = BASE_FEATURE_DIM * 2  # posición + velocidad
MASK_FEATURES = [197, 198, 199, 397, 398, 399]


def load_landmark_array(path):
    """Carga un .npy y valida la forma esperada (T, 1662)."""
    arr = np.load(path, allow_pickle=False)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'{path}: esperaba matriz 2D, recibí {arr.shape}')
    if arr.shape[1] != 1662 and arr.shape[0] == 1662:
        arr = arr.T
    if arr.shape[1] != 1662:
        raise ValueError(f'{path}: esperaba 1662 características por frame, recibí {arr.shape}')
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if len(arr) < 2:
        raise ValueError(f'{path}: secuencia demasiado corta ({len(arr)} frames)')
    return arr


def resample_frames(arr, target_len=SEQ_LEN):
    """Remuestreo lineal vectorizado a una longitud temporal fija."""
    n = len(arr)
    if n == target_len:
        return arr.astype(np.float32, copy=False)
    positions = np.linspace(0, n - 1, target_len, dtype=np.float32)
    lo = np.floor(positions).astype(np.int64)
    hi = np.minimum(lo + 1, n - 1)
    alpha = (positions - lo).astype(np.float32)[:, None]
    return (arr[lo] * (1.0 - alpha) + arr[hi] * alpha).astype(np.float32)


def landmarks_to_features(arr, target_len=SEQ_LEN):
    """Convierte landmarks crudos en una secuencia (target_len, 400)."""
    arr = resample_frames(arr, target_len)
    t = len(arr)

    pose = arr[:, POSE_SLICE].reshape(t, 33, 4)
    face = arr[:, FACE_SLICE].reshape(t, 468, 3)
    left_hand = arr[:, LEFT_HAND_SLICE].reshape(t, 21, 3)
    right_hand = arr[:, RIGHT_HAND_SLICE].reshape(t, 21, 3)

    # Centro del torso y escala robusta por video: distancia entre hombros.
    left_shoulder = pose[:, 11, :3]
    right_shoulder = pose[:, 12, :3]
    torso_center = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder[:, :2] - right_shoulder[:, :2], axis=1)
    valid_widths = shoulder_width[shoulder_width > 1e-4]
    body_scale = float(np.median(valid_widths)) if len(valid_widths) else 1.0
    body_scale = max(body_scale, 1e-3)

    def hand_features(hand):
        present = (np.max(np.abs(hand), axis=(1, 2)) > 1e-6).astype(np.float32)
        wrist = hand[:, 0, :]
        local = (hand - wrist[:, None, :]) / body_scale
        global_wrist = (wrist - torso_center) / body_scale
        local[present == 0] = 0.0
        global_wrist[present == 0] = 0.0
        return local.reshape(t, -1), global_wrist, present

    left_local, left_wrist, left_present = hand_features(left_hand)
    right_local, right_wrist, right_present = hand_features(right_hand)

    # Postura relativa al torso; se conserva visibilidad como cuarta característica.
    body_xyz = (pose[:, BODY_JOINTS, :3] - torso_center[:, None, :]) / body_scale
    body_visibility = pose[:, BODY_JOINTS, 3:4]
    body_features = np.concatenate([body_xyz, body_visibility], axis=2).reshape(t, -1)

    # Rasgos faciales compactos. Se conservan cambios locales respecto a la nariz.
    selected_face = face[:, FACE_JOINTS, :]
    nose = face[:, 1, :]
    eye_width = np.linalg.norm(face[:, 33, :2] - face[:, 263, :2], axis=1)
    valid_eye_widths = eye_width[eye_width > 1e-4]
    face_scale = float(np.median(valid_eye_widths)) if len(valid_eye_widths) else 1.0
    face_scale = max(face_scale, 1e-3)
    face_present = (np.max(np.abs(face), axis=(1, 2)) > 1e-6).astype(np.float32)
    face_local = (selected_face - nose[:, None, :]) / face_scale
    face_local[face_present == 0] = 0.0
    face_features = face_local.reshape(t, -1)

    # 126 mano + 6 trayectorias de muñeca + 32 cuerpo + 33 rostro + 3 máscaras = 200.
    base = np.concatenate([
        left_local, right_local,
        left_wrist, right_wrist,
        body_features, face_features,
        left_present[:, None], right_present[:, None], face_present[:, None]
    ], axis=1).astype(np.float32)

    if base.shape[1] != BASE_FEATURE_DIM:
        raise RuntimeError(f'Dimensión base inesperada: {base.shape[1]}')

    # Diferencias frame a frame. Las máscaras no representan velocidad.
    velocity = np.diff(base, axis=0, prepend=base[:1]).astype(np.float32)
    velocity[:, -3:] = 0.0
    features = np.concatenate([base, velocity], axis=1).astype(np.float32)
    if features.shape != (target_len, FEATURE_DIM):
        raise RuntimeError(f'Dimensión final inesperada: {features.shape}')
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)


print('Dimensión por frame:', FEATURE_DIM)
print('Máscaras de presencia:', MASK_FEATURES)

# Prueba rápida de procesamiento con un archivo real.
example_path = Path(metadata.iloc[0]['path'])
example_raw = load_landmark_array(example_path)
example_features = landmarks_to_features(example_raw)
print('Archivo de ejemplo:', example_path.name)
print('Forma original:', example_raw.shape)
print('Forma procesada:', example_features.shape)
print('Valores finitos:', np.isfinite(example_features).all())

## 4. Procesamiento del conjunto etiquetado

En esta celda se transforman los `.npy` del conjunto público de entrenamiento. Guardamos las características procesadas en `/kaggle/working` para facilitar la experimentación durante la sesión.

In [ ]:
def build_feature_tensor(meta_df, seq_len=SEQ_LEN):
    """Procesa las muestras etiquetadas y devuelve X, y."""
    sequences = []
    for row in tqdm(meta_df.itertuples(index=False), total=len(meta_df), desc='Procesando landmarks'):
        raw = load_landmark_array(row.path)
        sequences.append(landmarks_to_features(raw, seq_len))
    X_out = np.stack(sequences).astype(np.float32)
    y_out = meta_df['y'].to_numpy(dtype=np.int64)
    return X_out, y_out

X, y = build_feature_tensor(metadata)

print('X:', X.shape, X.dtype)
print('y:', y.shape, y.dtype)
print('Uso aproximado de memoria (MB):', round(X.nbytes / 1024**2, 1))
print('Distribución de etiquetas:', Counter(y.tolist()))

np.save(OUT_DIR / 'X_landmarks_features.npy', X)
np.save(OUT_DIR / 'y_labels.npy', y)
metadata.to_csv(OUT_DIR / 'train_metadata.csv', index=False)
print('Características y metadatos guardados en:', OUT_DIR)

## 5. Utilidades de escalado y métricas

El escalador se ajusta exclusivamente sobre las secuencias que pertenecen al entrenamiento de cada partición. Cada fold obtiene su propio promedio y desviación estándar.

In [ ]:
def fit_sequence_scaler(X_train):
    """Ajusta media y desviación por característica usando solo X_train."""
    flat = X_train.reshape(-1, X_train.shape[-1]).astype(np.float64)
    mean = flat.mean(axis=0).astype(np.float32)
    std = flat.std(axis=0).astype(np.float32)
    std[std < 1e-5] = 1.0
    return {'mean': mean, 'std': std}


def apply_sequence_scaler(X_data, scaler):
    return ((X_data - scaler['mean'][None, None, :]) /
            scaler['std'][None, None, :]).astype(np.float32)


def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }


def print_confusion(y_true, y_pred, title='Matriz de confusión'):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real')
    ax.set_xticks(range(5), CLASS_NAMES, rotation=35, ha='right')
    ax.set_yticks(range(5), CLASS_NAMES)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()
    return cm

# Diagnóstico: la longitud temporal ya quedó homogeneizada.
print('Forma de X:', X.shape)
print('Rango global de características:', float(X.min()), 'a', float(X.max()))

## 6. Línea base: SVM con resumen temporal

Antes de una red neuronal, entrenamos un SVM RBF con estadísticas temporales compactas (media, desviación, mínimo y máximo). Este baseline es rápido y nos indica si las características geométricas ya separan las clases.

La comparación se realiza con exactamente los mismos folds LOSO que utilizaremos para el TCN.

In [ ]:
def summarize_sequences(X_seq):
    """Resume cada secuencia sin perder por completo información de movimiento."""
    mean = X_seq.mean(axis=1)
    std = X_seq.std(axis=1)
    q10 = np.quantile(X_seq, 0.10, axis=1)
    q90 = np.quantile(X_seq, 0.90, axis=1)
    first = X_seq[:, 0, :]
    last = X_seq[:, -1, :]
    return np.concatenate([mean, std, q10, q90, first, last], axis=1).astype(np.float32)


def run_loso_svm(X_seq, y_all, meta_df):
    signers = sorted(meta_df['signer'].unique())
    X_summary = summarize_sequences(X_seq)
    rows, predictions = [], np.full(len(y_all), -1, dtype=np.int64)

    for heldout in signers:
        train_idx = np.where(meta_df['signer'].to_numpy() != heldout)[0]
        valid_idx = np.where(meta_df['signer'].to_numpy() == heldout)[0]
        model = Pipeline([
            ('scale', StandardScaler()),
            ('svc', SVC(C=2.0, kernel='rbf', class_weight='balanced'))
        ])
        model.fit(X_summary[train_idx], y_all[train_idx])
        pred = model.predict(X_summary[valid_idx])
        predictions[valid_idx] = pred
        metrics = calculate_metrics(y_all[valid_idx], pred)
        rows.append({'heldout_signer': heldout, 'n_valid': len(valid_idx), **metrics})
        print(f"{heldout}: accuracy={metrics['accuracy']:.4f} | macro-F1={metrics['macro_f1']:.4f}")

    return pd.DataFrame(rows), predictions

svm_results, svm_oof = run_loso_svm(X, y, metadata)
print('\nResumen SVM LOSO:')
display(svm_results)
print('Accuracy promedio:', round(svm_results.accuracy.mean(), 4),
      '+/-', round(svm_results.accuracy.std(ddof=0), 4))

## 7. Arquitectura TCN compacta

La red recibe una secuencia de 64 frames y 400 características por frame. Las convoluciones temporales dilatadas observan movimientos a distintas escalas. El bloque de atención resume los frames relevantes y el clasificador produce las cinco clases.

No usamos una red excesivamente profunda: el dataset contiene solo tres participantes públicos y el riesgo principal es el sobreajuste.

In [ ]:
class TemporalResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, dropout=0.25):
        super().__init__()
        padding = dilation
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.Dropout(dropout),
        )
        self.activation = nn.GELU()

    def forward(self, x):
        return self.activation(x + self.block(x))


# Los índices corresponden al orden creado en landmarks_to_features:
# manos (0:132), cuerpo (132:164), rostro (164:197), máscaras (197:200),
# seguidos por sus velocidades (200:400).
HAND_STREAM_IDX = list(range(0, 132)) + list(range(200, 332)) + [197, 198]
BODY_STREAM_IDX = list(range(132, 164)) + list(range(332, 364))
FACE_STREAM_IDX = list(range(164, 197)) + list(range(364, 397)) + [199]


class LSC5_TCN(nn.Module):
    """TCN multistream: codificadores separados y fusión temporal posterior."""
    def __init__(self, input_dim=FEATURE_DIM, channels=TCN_CHANNELS,
                 num_classes=5, dropout=DROPOUT):
        super().__init__()
        if input_dim != FEATURE_DIM:
            raise ValueError(f'input_dim debe ser {FEATURE_DIM}, recibí {input_dim}')
        if channels % 3 != 0:
            raise ValueError('TCN_CHANNELS debe ser múltiplo de 3 para las tres ramas.')

        self.register_buffer('hand_idx', torch.tensor(HAND_STREAM_IDX, dtype=torch.long))
        self.register_buffer('body_idx', torch.tensor(BODY_STREAM_IDX, dtype=torch.long))
        self.register_buffer('face_idx', torch.tensor(FACE_STREAM_IDX, dtype=torch.long))
        branch_channels = channels // 3

        def stream_encoder(input_features):
            return nn.Sequential(
                nn.Conv1d(input_features, branch_channels, kernel_size=1),
                nn.BatchNorm1d(branch_channels),
                nn.GELU(),
            )

        self.hand_encoder = stream_encoder(len(HAND_STREAM_IDX))
        self.body_encoder = stream_encoder(len(BODY_STREAM_IDX))
        self.face_encoder = stream_encoder(len(FACE_STREAM_IDX))

        self.temporal = nn.Sequential(*[
            TemporalResidualBlock(channels, dilation=d, dropout=dropout)
            for d in [1, 2, 4, 8]
        ])
        self.attention = nn.Conv1d(channels, 1, kernel_size=1)
        self.classifier = nn.Sequential(
            nn.Linear(channels * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (batch, tiempo, características). Cada rama recibe su propio subconjunto.
        hand = x.index_select(2, self.hand_idx).transpose(1, 2)
        body = x.index_select(2, self.body_idx).transpose(1, 2)
        face = x.index_select(2, self.face_idx).transpose(1, 2)

        # Codificadores anatómicos independientes.
        streams = torch.cat([
            self.hand_encoder(hand),
            self.body_encoder(body),
            self.face_encoder(face),
        ], dim=1)

        # Fusión de ramas y aprendizaje de dependencias temporales multiescala.
        features = self.temporal(streams)
        weights = torch.softmax(self.attention(features), dim=-1)
        attended = torch.sum(features * weights, dim=-1)
        maximum = torch.max(features, dim=-1).values
        pooled = torch.cat([attended, maximum], dim=1)
        return self.classifier(pooled)


model_test = LSC5_TCN().to(DEVICE)
params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
with torch.no_grad():
    dummy = torch.zeros(2, SEQ_LEN, FEATURE_DIM, device=DEVICE)
    dummy_out = model_test(dummy)
print('Dimensiones de ramas:', len(HAND_STREAM_IDX), len(BODY_STREAM_IDX), len(FACE_STREAM_IDX))
print('Parámetros entrenables:', f'{params:,}')
print('Salida de prueba:', tuple(dummy_out.shape))
del model_test, dummy, dummy_out

## 8. Entrenamiento, aumento de datos y selección interna de épocas

El aumento de datos es deliberadamente moderado: pequeñas deformaciones temporales y ruido gaussiano bajo sobre características continuas. No invertimos izquierda/derecha automáticamente porque la lateralidad puede tener valor lingüístico.

**Validación anidada:** para cada participante reservado por LOSO, la selección del número de épocas se realiza con un split interno agrupado por `group_id` dentro de los otros dos participantes. Después se crea un modelo nuevo y se entrena con todos los ejemplos de esos dos participantes durante el número de épocas seleccionado. El participante exterior solo se usa para calcular la métrica final del fold.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X_data, y_data, augment=False):
        self.X = torch.from_numpy(np.asarray(X_data, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y_data, dtype=np.int64))
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def _augment_sequence(self, x):
        # Deformación temporal monotónica suave.
        if random.random() < 0.65:
            t = x.shape[0]
            gamma = random.uniform(0.90, 1.10)
            positions = (np.linspace(0, 1, t, dtype=np.float32) ** gamma) * (t - 1)
            lo = np.floor(positions).astype(np.int64)
            hi = np.minimum(lo + 1, t - 1)
            alpha = (positions - lo)[:, None]
            x = x[lo] * (1.0 - alpha) + x[hi] * alpha

        # Ruido pequeño solo en variables continuas, no en máscaras binarias.
        if random.random() < 0.70:
            noise = np.random.normal(0, 0.012, size=x.shape).astype(np.float32)
            noise[:, MASK_FEATURES] = 0.0
            x = x + noise
        return x.astype(np.float32)

    def __getitem__(self, idx):
        x = self.X[idx].numpy().copy()
        if self.augment:
            x = self._augment_sequence(x)
        return torch.from_numpy(x), self.y[idx]


def make_loader(X_data, y_data, train=False):
    dataset = SequenceDataset(X_data, y_data, augment=train)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
        drop_last=False,
    )


def predict_tcn(model, X_scaled, batch_size=64):
    model.eval()
    outputs = []
    dummy_y = np.zeros(len(X_scaled), dtype=np.int64)
    loader = DataLoader(SequenceDataset(X_scaled, dummy_y, augment=False),
                        batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            outputs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(outputs, axis=0) if outputs else np.empty((0, 5), dtype=np.float32)


def fit_tcn(X_train_raw, y_train, X_valid_raw=None, y_valid=None,
            max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED):
    """Ajusta scaler solo con train y entrena TCN. Si hay validación, aplica early stopping."""
    seed_everything(seed)
    scaler = fit_sequence_scaler(X_train_raw)
    X_train = apply_sequence_scaler(X_train_raw, scaler)
    train_loader = make_loader(X_train, y_train, train=True)

    has_valid = X_valid_raw is not None and y_valid is not None
    if has_valid:
        X_valid = apply_sequence_scaler(X_valid_raw, scaler)
        valid_loader = make_loader(X_valid, y_valid, train=False)

    model = LSC5_TCN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
    amp_enabled = bool(USE_AMP and DEVICE.type == 'cuda')
    try:
        grad_scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    except (AttributeError, TypeError):
        grad_scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    best_epoch = max_epochs
    best_accuracy = -1.0
    best_loss = float('inf')
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, seen = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=amp_enabled):
                logits = model(xb)
                loss = criterion(logits, yb)
            grad_scaler.scale(loss).backward()
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            grad_scaler.step(optimizer)
            grad_scaler.update()
            running_loss += float(loss.item()) * len(yb)
            seen += len(yb)

        train_loss = running_loss / max(seen, 1)
        row = {'epoch': epoch, 'train_loss': train_loss}

        if has_valid:
            model.eval()
            val_loss_sum, val_seen, val_pred, val_true = 0.0, 0, [], []
            with torch.no_grad():
                for xb, yb in valid_loader:
                    xb = xb.to(DEVICE, non_blocking=True)
                    yb = yb.to(DEVICE, non_blocking=True)
                    with torch.autocast(device_type=DEVICE.type, enabled=amp_enabled):
                        logits = model(xb)
                        loss = criterion(logits, yb)
                    val_loss_sum += float(loss.item()) * len(yb)
                    val_seen += len(yb)
                    val_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())
                    val_true.extend(yb.cpu().numpy().tolist())

            val_loss = val_loss_sum / max(val_seen, 1)
            val_acc = accuracy_score(val_true, val_pred)
            row.update({'val_loss': val_loss, 'val_accuracy': val_acc})
            improved = (val_acc > best_accuracy + 1e-6) or (
                abs(val_acc - best_accuracy) <= 1e-6 and val_loss < best_loss
            )
            if improved:
                best_accuracy, best_loss = float(val_acc), float(val_loss)
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
        else:
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history.append(row)
        scheduler.step()

        if has_valid and epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, scaler, int(best_epoch), pd.DataFrame(history)


def make_inner_group_split(global_train_idx, meta_df, y_all, seed=SEED):
    """Busca un split agrupado por ejecución que conserve las cinco clases."""
    local_y = y_all[global_train_idx]
    local_groups = meta_df.iloc[global_train_idx]['group_id'].to_numpy()
    all_classes = set(range(len(CLASS_NAMES)))

    splitter = GroupShuffleSplit(n_splits=80, test_size=0.18, random_state=seed)
    for tr_pos, va_pos in splitter.split(np.zeros(len(global_train_idx)), local_y, local_groups):
        if set(local_y[tr_pos]) == all_classes and set(local_y[va_pos]) == all_classes:
            return global_train_idx[tr_pos], global_train_idx[va_pos], 'GroupShuffleSplit'

    # Fallback documentado: mantiene estratificación si no hay suficientes grupos por clase.
    warnings.warn('No se encontró split agrupado con las cinco clases; se usa split estratificado interno.')
    fallback = StratifiedShuffleSplit(n_splits=1, test_size=0.18, random_state=seed)
    tr_pos, va_pos = next(fallback.split(np.zeros(len(local_y)), local_y))
    return global_train_idx[tr_pos], global_train_idx[va_pos], 'StratifiedShuffleSplit fallback'

print('Funciones de entrenamiento definidas. AMP:', USE_AMP and DEVICE.type == 'cuda')

## 9. Validación LOSO anidada del TCN

Cada fold reserva un participante completo. El split interno sirve únicamente para decidir el número de épocas. Después el modelo se reinicia, ajusta su escalador con los dos participantes públicos del fold y se entrena con todos ellos durante las épocas seleccionadas.

Así, las muestras del participante exterior no participan en el escalado, early stopping ni actualización de pesos.

In [ ]:
def run_loso_tcn(X_seq, y_all, meta_df):
    signers = sorted(meta_df['signer'].unique())
    if len(signers) < 3:
        warnings.warn(f'Se detectaron {len(signers)} participantes; LOSO se ejecutará con los disponibles.')

    signer_values = meta_df['signer'].to_numpy()
    fold_rows = []
    oof_pred = np.full(len(y_all), -1, dtype=np.int64)
    best_epochs = []
    fold_histories = {}

    for fold_no, heldout in enumerate(signers, start=1):
        print('\n' + '=' * 72)
        print(f'FOLD {fold_no}/{len(signers)} | participante reservado: {heldout}')
        outer_train_idx = np.where(signer_values != heldout)[0]
        outer_valid_idx = np.where(signer_values == heldout)[0]

        inner_train_idx, inner_valid_idx, split_name = make_inner_group_split(
            outer_train_idx, meta_df, y_all, seed=SEED + fold_no
        )
        print(f'Entrenamiento interno: {len(inner_train_idx)} | validación interna: {len(inner_valid_idx)} ({split_name})')

        # 1) Selección de épocas usando solo los participantes de entrenamiento.
        _, _, best_epoch, history = fit_tcn(
            X_seq[inner_train_idx], y_all[inner_train_idx],
            X_seq[inner_valid_idx], y_all[inner_valid_idx],
            max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED + fold_no
        )
        best_epochs.append(best_epoch)
        fold_histories[heldout] = history
        print('Mejor época interna:', best_epoch)

        # 2) Reentrenamiento desde cero con los dos participantes completos.
        fold_model, fold_scaler, _, _ = fit_tcn(
            X_seq[outer_train_idx], y_all[outer_train_idx],
            X_valid_raw=None, y_valid=None,
            max_epochs=best_epoch, patience=best_epoch, seed=SEED + 100 + fold_no
        )

        # 3) Evaluación una sola vez sobre el participante reservado.
        X_outer_valid = apply_sequence_scaler(X_seq[outer_valid_idx], fold_scaler)
        probabilities = predict_tcn(fold_model, X_outer_valid)
        predictions = probabilities.argmax(axis=1)
        oof_pred[outer_valid_idx] = predictions
        metrics = calculate_metrics(y_all[outer_valid_idx], predictions)
        fold_rows.append({
            'heldout_signer': heldout,
            'n_train': len(outer_train_idx),
            'n_valid': len(outer_valid_idx),
            'inner_split': split_name,
            'best_epochs': best_epoch,
            **metrics,
        })
        print(f"Resultado {heldout}: accuracy={metrics['accuracy']:.4f} | macro-F1={metrics['macro_f1']:.4f}")
        print(classification_report(y_all[outer_valid_idx], predictions,
                                    labels=list(range(5)), target_names=CLASS_NAMES,
                                    zero_division=0))
        print_confusion(y_all[outer_valid_idx], predictions, f'LOSO - reservado {heldout}')

        del fold_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results = pd.DataFrame(fold_rows)
    if np.any(oof_pred < 0):
        warnings.warn('Hay observaciones sin predicción OOF; revisa el inventario de participantes.')
    return results, oof_pred, best_epochs, fold_histories

# Ejecuta los tres folds. En GPU T4/P100 es razonable; en CPU reduce MAX_EPOCHS y TCN_CHANNELS.
tcn_results, tcn_oof, best_epochs, tcn_histories = run_loso_tcn(X, y, metadata)

print('\nResumen TCN LOSO:')
display(tcn_results)
print('Accuracy promedio:', round(tcn_results.accuracy.mean(), 4),
      '+/-', round(tcn_results.accuracy.std(ddof=0), 4))
print('Épocas seleccionadas:', best_epochs)
print('Accuracy OOF agregado:', round(accuracy_score(y, tcn_oof), 4))
print('Macro-F1 OOF agregado:', round(f1_score(y, tcn_oof, average='macro'), 4))
print_confusion(y, tcn_oof, 'Matriz de confusión OOF agregada')

## 10. Comparación y decisión del modelo

La métrica principal es Accuracy Top-1. Macro-F1, desempeño por participante y matrices de confusión ayudan a identificar fallos concentrados.

No debemos escoger un modelo por una única partición. Revisamos promedio y dispersión entre los tres participantes. El SVM es un control: si supera al TCN, no se debe forzar la red neuronal; se debe investigar por qué las características resumen generalizan mejor.

In [ ]:
comparison = pd.DataFrame([
    {'model': 'SVM-RBF',
     'mean_accuracy': svm_results.accuracy.mean(),
     'std_accuracy': svm_results.accuracy.std(ddof=0),
     'mean_macro_f1': svm_results.macro_f1.mean()},
    {'model': 'TCN',
     'mean_accuracy': tcn_results.accuracy.mean(),
     'std_accuracy': tcn_results.accuracy.std(ddof=0),
     'mean_macro_f1': tcn_results.macro_f1.mean()},
]).sort_values('mean_accuracy', ascending=False)
display(comparison)

print('Accuracy por participante:')
display(tcn_results[['heldout_signer', 'accuracy', 'macro_f1', 'best_epochs']])

# Regla de decisión: la opción con mayor accuracy media LOSO es la referencia.
# Si hay diferencias mínimas, considerar estabilidad (menor desviación) y macro-F1.
if tcn_results.accuracy.mean() >= svm_results.accuracy.mean():
    print('El TCN iguala o supera el baseline en accuracy media LOSO. Continuar con TCN.')
else:
    print('El SVM supera al TCN en accuracy media LOSO. Revisar el modelo y no asumir que la red es mejor.')

# Guardar resultados y predicciones OOF para auditoría/reporte.
svm_results.to_csv(OUT_DIR / 'loso_svm_results.csv', index=False)
tcn_results.to_csv(OUT_DIR / 'loso_tcn_results.csv', index=False)
pd.DataFrame({'video_id': metadata['video_id'], 'signer': metadata['signer'],
              'true_sign': metadata['label'],
              'predicted_sign': [ID_TO_CLASS[int(v)] for v in tcn_oof]}).to_csv(
    OUT_DIR / 'tcn_oof_predictions.csv', index=False
)
with open(OUT_DIR / 'selected_epochs.json', 'w', encoding='utf-8') as f:
    json.dump({'best_epochs_by_fold': [int(v) for v in best_epochs],
               'median_epochs': int(np.median(best_epochs))}, f, indent=2)
print('Resultados guardados en:', OUT_DIR)

## 11. Entrenamiento final con los tres participantes públicos

Una vez terminada la comparación LOSO y congelados los hiperparámetros, entrenamos un modelo final con **todos los ejemplos etiquetados disponibles**. El número de épocas se fija con la mediana de las épocas seleccionadas en los folds internos.

Este modelo queda listo para inferencia. No se utiliza el test para modificar pesos, normalización, umbrales o hiperparámetros.

In [ ]:
FINAL_EPOCHS = max(1, int(np.median(best_epochs)))
print('Épocas finales (mediana de LOSO):', FINAL_EPOCHS)

final_model, final_scaler, _, final_history = fit_tcn(
    X, y,
    X_valid_raw=None, y_valid=None,
    max_epochs=FINAL_EPOCHS, patience=FINAL_EPOCHS,
    seed=SEED + 999
)

# Guardar pesos, escalador y configuración para reproducir inferencia.
checkpoint = {
    'model_state_dict': final_model.state_dict(),
    'scaler_mean': final_scaler['mean'],
    'scaler_std': final_scaler['std'],
    'config': {
        'seq_len': SEQ_LEN,
        'feature_dim': FEATURE_DIM,
        'base_feature_dim': BASE_FEATURE_DIM,
        'channels': TCN_CHANNELS,
        'dropout': DROPOUT,
        'classes': CLASS_NAMES,
        'final_epochs': FINAL_EPOCHS,
        'seed': SEED,
    }
}
torch.save(checkpoint, OUT_DIR / 'lsc5_tcn_final.pt')
final_history.to_csv(OUT_DIR / 'final_training_history.csv', index=False)
with open(OUT_DIR / 'model_config.json', 'w', encoding='utf-8') as f:
    json.dump(checkpoint['config'], f, indent=2, ensure_ascii=False)

print('Modelo final guardado:', OUT_DIR / 'lsc5_tcn_final.pt')
print('Artefactos disponibles:')
for p in sorted(OUT_DIR.iterdir()):
    print('-', p.name, f'({p.stat().st_size / 1024**2:.2f} MB)')

## 12. Inferencia final sobre test (opcional y estrictamente separada)

**No ejecutar hasta congelar el modelo y cerrar la selección de hiperparámetros.** Esta celda no calcula métricas, no adapta el escalador y no modifica el modelo. Solo aplica la transformación determinista ya definida y genera una predicción por ID.

Por defecto `RUN_FINAL_TEST_INFERENCE = False`. Para generar `submission.csv`, cambia esa variable a `True` y ejecuta únicamente después de completar las fases anteriores.

In [ ]:
def discover_test_landmarks(root=DATA_ROOT, override=None):
    if override is not None:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(f'La ruta TEST_LANDMARKS_DIR no existe: {p}')
        return p
    candidates = []
    for p in root.rglob('landmarks'):
        if not p.is_dir():
            continue
        if 'test' in [part.lower() for part in p.parts] and any(p.rglob('*.npy')):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError('No encontré test/landmarks. Define TEST_LANDMARKS_DIR manualmente.')
    candidates.sort(key=lambda p: len(str(p)))
    return candidates[0]


def build_test_features(test_dir):
    paths = sorted(Path(test_dir).rglob('*.npy'))
    if not paths:
        raise RuntimeError(f'No hay .npy en {test_dir}')
    ids, feats = [], []
    for p in tqdm(paths, desc='Inferencia: transformando test'):
        ids.append(p.stem)
        feats.append(landmarks_to_features(load_landmark_array(p), SEQ_LEN))
    return pd.DataFrame({'video_id': ids, 'path': [str(p) for p in paths]}), np.stack(feats).astype(np.float32)


def generate_submission(model, scaler, test_dir, out_dir=OUT_DIR):
    """Solo inferencia. No usa etiquetas ni estadísticas del conjunto de test."""
    test_meta, X_test = build_test_features(test_dir)
    X_test_scaled = apply_sequence_scaler(X_test, scaler)  # escalador congelado del train completo
    probabilities = predict_tcn(model, X_test_scaled)
    pred_ids = probabilities.argmax(axis=1)
    submission = pd.DataFrame({
        'video_id': test_meta['video_id'],
        'predicted_sign': [ID_TO_CLASS[int(i)] for i in pred_ids]
    })

    # Si existe plantilla, se respeta exactamente su orden e IDs.
    templates = list(DATA_ROOT.rglob('submission_template.csv'))
    if templates:
        template = pd.read_csv(templates[0])
        if 'video_id' not in template.columns:
            raise ValueError('La plantilla no contiene columna video_id.')
        submission = template[['video_id']].merge(submission, on='video_id', how='left', validate='one_to_one')
        if submission['predicted_sign'].isna().any():
            raise ValueError('Hay IDs de plantilla sin predicción. Revisa los archivos de test.')

    if submission['video_id'].duplicated().any():
        raise ValueError('Hay video_id duplicados en la entrega.')
    if submission['predicted_sign'].isna().any():
        raise ValueError('Hay predicciones vacías.')
    if not set(submission['predicted_sign']).issubset(set(CLASS_NAMES)):
        raise ValueError('Se generó una etiqueta no permitida.')

    output_path = Path(out_dir) / 'submission.csv'
    submission.to_csv(output_path, index=False, encoding='utf-8')
    print('CSV generado:', output_path)
    print('Filas:', len(submission))
    print('Columnas:', submission.columns.tolist())
    display(submission.head())
    return submission


if RUN_FINAL_TEST_INFERENCE:
    TEST_LANDMARKS_DIR = discover_test_landmarks(DATA_ROOT, TEST_LANDMARKS_DIR)
    submission = generate_submission(final_model, final_scaler, TEST_LANDMARKS_DIR)
else:
    print('Inferencia de test desactivada. No se ha leído ningún archivo de test.')

## 13. Checklist antes de entregar

- [ ] Se verificó que `train/landmarks` contiene únicamente los tres participantes públicos.
- [ ] Se inspeccionaron formas, valores faltantes y correspondencia de los `.npy`.
- [ ] Los tres folds LOSO reservaron un participante completo.
- [ ] Los grupos de vistas de una ejecución se mantuvieron unidos en la validación interna cuando fue posible.
- [ ] El escalador se ajustó solo con el subconjunto de entrenamiento de cada partición.
- [ ] Se registraron Accuracy Top-1 y macro-F1 por participante.
- [ ] Se comparó TCN contra SVM sin mirar resultados de test.
- [ ] Se congelaron hiperparámetros antes de ejecutar inferencia final.
- [ ] El CSV final tiene `video_id,predicted_sign`, sin índice, duplicados ni IDs faltantes.
- [ ] Se conserva el notebook, los pesos, la configuración y los resultados OOF para auditoría.

## Ajustes para el servidor UNAB

Al pasar al servidor, conserva la misma metodología y cambia únicamente recursos/hyperparámetros en una nueva corrida versionada. Puedes probar `TCN_CHANNELS=144`, `MAX_EPOCHS=80` y `SEQ_LEN=96`, pero cada cambio debe volver a evaluarse con los tres folds LOSO. No adoptes el modelo más grande salvo que mejore el promedio y no deteriore notablemente el peor participante.

**Importante:** el número final de épocas y cualquier mejora deben salir de experimentos ejecutados. Este notebook no presupone ni promete una accuracy específica.